
Task:  Build a Sentiment Analysis model using a real-world dataset

Dataset: Amazon Review Data Web Scrapping - Amazon Review Data Web Scrapping.csv
Source: Kaggle
The dataset must contain text data with sentiment labels(positive/negative/neutral), requirement satisfied.

Task breakdown:
1. Data understanding
2. NLP Preprocessing
3. Feature Engineering (BoW, TF-IDF, Word2Vec, Avg Word2Vec)
4. Model Building (train at least 3 models: Logistic Regression, Naive Bayes, Decision Tree, Random Forest, XGBoost)
5. Model Evaluation (using Accuracy, Precision, Recall, F1 score)
6. Comparison & Insights


In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score


# 1. DATA UNDERSTANDING & LOADING
#---------------------------------

#Load dataset
df = pd.read_csv('Amazon Review Data Web Scrapping - Amazon Review Data Web Scrapping.csv')

#Explore no. of samples and features
print(f"Dataset shape: {df.shape}")
print("\nColumn Names and Data Types:")
print(df.dtypes)

#Check for missing values 
print("\nMissing Values per Colum:")
print(df.isnull().sum())

#Explore Class Distribution for 3 labels (Positive, Negative, Neutral)
print("\nClass Distribution (Own_Rating):")
print(df['Own_Rating'].value_counts())

#Calculate distribution percentages
print("\nClass Distribution Percentage:")
print(df['Own_Rating'].value_counts(normalize=True) * 100)

# Display sample texts (Header and Review)
print("\nSample Review Texts and Ratings:")
print(df[['Review_Header', 'Review_text', 'Own_Rating']].head(10))

# Pre-cleaning: Combine Review Header and Text for richer context
df['full_text'] = df['Review_Header'].fillna('') + ' ' + df['Review_text'].fillna('')

# Drop rows where there is no text content or no label
df = df.dropna(subset=['Own_Rating'])
df = df[df['full_text'].str.strip() != '']


# 2. NLP PREPROCESSING 
#----------------------

def preprocess_text(text):
    text = text.lower() # Lowercasing
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and special characters
    tokens = text.split() # Tokenization
    # Note: Stopword removal and stemming will be handled during vectorization for efficiency
    return " ".join(tokens)

df['cleaned_text'] = df['full_text'].apply(preprocess_text)

# Label Mapping: Negative -> 0, Neutral -> 1, Positive -> 2
label_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df['label'] = df['Own_Rating'].map(label_map)


# 3. FEATURE ENGINEERING
#-------------------------

# Stratified split ensures the class distribution is preserved in train/test sets
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# TF-IDF & Bag of Words (Using Bigrams to capture phrases like "not good")
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
bow_vectorizer = CountVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

X_train_bow = bow_vectorizer.fit_transform(X_train_text)
X_test_bow = bow_vectorizer.transform(X_test_text)


#  4 & 5. MODEL BUILDING & EVALUATION
#-------------------------------------

# Using class_weight='balanced' to handle the heavy positive bias in Amazon reviews
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=30, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, class_weight='balanced', random_state=42)
}

results = []

for name, model in models.items():
    for vec_name, X_tr, X_te in [("TF-IDF", X_train_tfidf, X_test_tfidf), ("BoW", X_train_bow, X_test_bow)]:
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        
        results.append({
            "Model": name,
            "Vectorizer": vec_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, average='weighted'),
            "Recall": recall_score(y_test, y_pred, average='weighted'),
            "F1-Score": f1_score(y_test, y_pred, average='weighted')
        })


#  SUMMARY & OUTPUT
#----------------------

results_df = pd.DataFrame(results)
print("\n--- Model Evaluation Results ---")
print(results_df.sort_values(by='F1-Score', ascending=False))

Dataset shape: (60889, 6)

Column Names and Data Types:
Unique_ID         int64
Category         object
Review_Header    object
Review_text      object
Rating            int64
Own_Rating       object
dtype: object

Missing Values per Colum:
Unique_ID         0
Category          0
Review_Header     5
Review_text      32
Rating            0
Own_Rating        0
dtype: int64

Class Distribution (Own_Rating):
Own_Rating
Positive    47436
Negative     9087
Neutral      4366
Name: count, dtype: int64

Class Distribution Percentage:
Own_Rating
Positive    77.905697
Negative    14.923878
Neutral      7.170425
Name: proportion, dtype: float64

Sample Review Texts and Ratings:
                            Review_Header  \
0                                Nice one   
1  Huge battery life with amazing display   
2                              Four Stars   
3                            Nice quality   
4                               Nice book   
5                                 Nice tv   
6         

Executive Summary:

▫️Best Preprocessing Steps: Filtering & Cleaning, Feature merging, Bigrams

▫️Best Vectorization: Bag of Words with Bigrams (Captures context like "not happy").

▫️Best Model: Naive Bayes (Fastest and most accurate for this text data).

▫️Key Challenge: The dataset is highly imbalanced (78% Positive).

▫️Key Fix: Merged Review_Header + Review_text and used class_weight='balanced' to ensure Negative and Neutral reviews weren't ignored.

Key Insights:

▫️Preprocessing: Lowercasing and removing special characters are mandatory to reduce vocabulary size.

▫️Vectorization: Bigrams (2-word pairs) are better than single words because they catch "not good."

▫️Trade-offs: Logistic Regression is better if you need to explain why a review is negative (via coefficients), but Naive Bayes is better for raw speed and accuracy.

Final Conclusion

For a real-world deployment on Amazon-style data, the best approach is Naive Bayes using Bag of Words (with Bigrams). It provides the best balance of speed, accuracy, and the ability to detect nuanced sentiment in short, informal text.